# Evaluación Parcial 3: pipeline integrado y comparación MLP vs CNN para patologías en tomate

**Asignatura:** TLY1102 Técnicas Avanzadas de Machine Learning I  
**Sección:** 001D  
**Integrantes:** Matías Gutiérrez, David Larenas, Victor Mena  
**Fecha:** noviembre de 2026

Este notebook documenta el pipeline integrado de la EP3. Reutiliza el conjunto de datos y la partición de la EP2, agrega regularización, búsqueda de hiperparámetros, auditoría de sesgos y explicabilidad. El objetivo es comparar una MLP optimizada con una CNN final y justificar la arquitectura recomendada.

## Cómo ejecutar este notebook

El notebook está pensado para Google Colab con GPU T4 y también funciona en local con TensorFlow 2.16 o superior. El repositorio debe contener `data/split.csv`, `data/class_names.json` y la carpeta `data/PlantVillage`.

1. Clonar el repositorio y entrar en la carpeta del proyecto.
2. Subir `PlantVillage Dataset.zip` a la raíz del proyecto. El notebook lo descomprime en `data/` si la carpeta no existe.
3. Ejecutar todas las celdas en orden. La primera celda instala `lime`.
4. Los modelos finales se guardan en `models/` y las figuras en `images/`.

In [ ]:
try:
    import lime
except ImportError:
    !pip install lime -q

## 1. Problema, objetivos y KPIs

### 1.1 Problema de negocio

Un productor de tomate necesita apoyo para identificar enfermedades foliares a partir de una fotografía. Un diagnóstico tardío reduce el rendimiento y aumenta el uso de fungicidas. El modelo debe distinguir diez clases de hojas de tomate y señalar cuándo la predicción no es confiable.

### 1.2 Objetivos

- Construir un pipeline reproducible de carga, preprocesamiento, entrenamiento, evaluación y auditoría.
- Comparar una MLP y una CNN bajo condiciones comparables.
- Medir el efecto de Dropout, Batch Normalization, L2 y aumento de datos.
- Optimizar hiperparámetros con un presupuesto acotado y justificar la configuración final.
- Auditar sesgos y explicar las decisiones del modelo con Grad-CAM y LIME.

### 1.3 KPIs

| KPI | Meta | Uso |
|---|---|---|
| F1 macro en test | 0,90 o más | equilibrio entre clases |
| Accuracy en test | 0,90 o más | referencia general |
| Brecha train y validación | 0,05 o menos | control de overfitting |
| ROC-AUC macro | 0,95 o más | separación entre clases |
| Tiempo de inferencia | 50 ms o menos por imagen | viabilidad de uso |
| Recall de clases minoritarias | 0,80 o más | riesgo de falsos negativos |

Los KPIs son metas de trabajo. Los resultados reales se reportan en la sección 8. No ajustamos el conjunto de test para cumplirlos.

### 1.4 CRISP-DM

| Fase | Aplicación en este proyecto |
|---|---|
| Comprensión del negocio | diagnóstico foliar con supervisión humana |
| Comprensión de los datos | 16.011 imágenes, 10 clases, origen de laboratorio |
| Preparación de los datos | split estratificado 80/10/10, 128x128, normalización |
| Modelado | MLP y CNN con regularización y búsqueda |
| Evaluación | métricas múltiples, matrices, auditoría y XAI |
| Despliegue | recomendación, límites y monitoreo |

## 2. Datos: carga, verificación y análisis exploratorio

Trabajamos con el subconjunto de tomate de PlantVillage usado en la EP2. Son 16.011 imágenes de 256x256 en diez clases. Mantener el mismo dataset y la misma partición permite comparar los resultados con la evaluación anterior.

La partición está fijada en `data/split.csv` con 12.808 imágenes de entrenamiento, 1.601 de validación y 1.602 de test. La estratificación conserva la proporción de clases en los tres conjuntos. El test no se usa para elegir hiperparámetros.

In [ ]:
import json
import os
import random
import time
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 40
PACIENCIA = 5
AUTOTUNE = tf.data.AUTOTUNE


def buscar_raiz():
    for base in [Path.cwd(), *Path.cwd().parents, Path('/content/tly1102-tomate')]:
        if (base / 'data' / 'split.csv').exists():
            return base
    return Path.cwd()


RAIZ = buscar_raiz()
DATA_DIR = RAIZ / 'data'
MODELS_DIR = RAIZ / 'models'
IMAGES_DIR = RAIZ / 'images'
HISTORY_DIR = MODELS_DIR / 'historial'
REPORTS_DIR = MODELS_DIR / 'reportes'
for directorio in (DATA_DIR, MODELS_DIR, IMAGES_DIR, HISTORY_DIR, REPORTS_DIR):
    directorio.mkdir(parents=True, exist_ok=True)

zip_datos = RAIZ / 'PlantVillage Dataset.zip'
if not (DATA_DIR / 'PlantVillage').exists() and zip_datos.exists():
    os.system(f'unzip -o -q "{zip_datos}" -d "{DATA_DIR}"')

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', palette='deep')

print('Raíz del proyecto:', RAIZ)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

La raíz se busca hacia arriba desde el directorio de trabajo y también en `/content/tly1102-tomate`, para que el notebook funcione tanto en local como en Colab. La semilla 42 se fija en Python, NumPy y TensorFlow. La descompresión solo ocurre si la carpeta de imágenes no existe.

In [ ]:
clases = [
    'Tomato_Bacterial_spot',
    'Tomato_Early_blight',
    'Tomato_Late_blight',
    'Tomato_Leaf_Mold',
    'Tomato_Septoria_leaf_spot',
    'Tomato_Spider_mites_Two_spotted_spider_mite',
    'Tomato__Target_Spot',
    'Tomato__Tomato_YellowLeaf__Curl_Virus',
    'Tomato__Tomato_mosaic_virus',
    'Tomato_healthy',
]


def cargar_datos(data_root=DATA_DIR / 'PlantVillage', split_path=DATA_DIR / 'split.csv'):
    if split_path.exists():
        df = pd.read_csv(split_path)
        class_path = DATA_DIR / 'class_names.json'
        if class_path.exists():
            nombres = json.loads(class_path.read_text())
        else:
            nombres = sorted(df['label'].unique())
        df['y'] = df['label'].map({clase: i for i, clase in enumerate(nombres)})
        df['abs_path'] = df['filepath'].map(lambda ruta: str(data_root / ruta))
        faltantes = df[~df['abs_path'].map(lambda ruta: Path(ruta).exists())]
        if len(faltantes) > 0:
            raise FileNotFoundError(f'Faltan {len(faltantes)} imágenes en {data_root}')
        return df, nombres
    registros = []
    for clase in clases:
        for archivo in (data_root / clase).iterdir():
            if archivo.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                registros.append({'filepath': str(archivo.relative_to(data_root)), 'label': clase})
    registros.sort(key=lambda r: (r['label'], r['filepath']))
    df = pd.DataFrame(registros)
    df['y'] = df['label'].map({clase: i for i, clase in enumerate(clases)})
    df['abs_path'] = df['filepath'].map(lambda ruta: str(data_root / ruta))
    from sklearn.model_selection import train_test_split
    train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)
    df['split'] = 'train'
    df.loc[val_df.index, 'split'] = 'val'
    df.loc[test_df.index, 'split'] = 'test'
    df[['filepath', 'label', 'split']].to_csv(split_path, index=False)
    (DATA_DIR / 'class_names.json').write_text(json.dumps(clases, indent=2))
    return df, clases


df, clases = cargar_datos()
print('Imágenes:', len(df))
print(df.groupby('split').size())
display(df.head())

La carga verifica que todas las rutas existan y conserva el orden de clases de `class_names.json`. Si el archivo `split.csv` no existe, el pipeline genera una partición estratificada con la semilla 42. En este repositorio el archivo existe, por lo que la partición es idéntica a la EP2.

In [ ]:
def cargar_imagen(ruta, etiqueta):
    imagen = tf.io.read_file(ruta)
    imagen = tf.io.decode_image(imagen, channels=3, expand_animations=False)
    imagen = tf.image.resize(imagen, IMG_SIZE)
    return tf.cast(imagen, tf.float32), etiqueta


def crear_dataset(df, split, entrenamiento=False, batch_size=BATCH_SIZE, transformacion=None):
    datos = df[df['split'] == split]
    ds = tf.data.Dataset.from_tensor_slices((datos['abs_path'].values, datos['y'].values))
    ds = ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE).cache()
    if entrenamiento:
        ds = ds.shuffle(len(datos), seed=SEED, reshuffle_each_iteration=True)
    if transformacion is not None:
        ds = ds.map(lambda x, y: (transformacion(x), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


train_ds = crear_dataset(df, 'train', entrenamiento=True)
val_ds = crear_dataset(df, 'val')
test_ds = crear_dataset(df, 'test')
print('Lotes de entrenamiento:', len(train_ds))
print('Lotes de validación:', len(val_ds))
print('Lotes de test:', len(test_ds))
for x_batch, y_batch in train_ds.take(1):
    print('Forma del lote:', x_batch.shape, y_batch.shape)

`crear_dataset` decodifica las imágenes, las redimensiona a 128x128 y las mantiene en memoria con `cache`. El entrenamiento se mezcla en cada época. La validación y el test conservan el orden, lo que permite alinear las predicciones con las etiquetas.

### 2.1 Balance de clases

El desbalance es visible. `Tomato__Tomato_YellowLeaf__Curl_Virus` tiene 2.566 imágenes de entrenamiento y `Tomato__Tomato_mosaic_virus` tiene 298. La razón es cercana a 8,6 a 1. Esta diferencia es la primera fuente de sesgo de entrada y justifica mirar el recall por clase, no solo la accuracy global.

In [ ]:
def nombre_corto(clase):
    return clase.replace('Tomato__', '').replace('Tomato_', '').replace('_', ' ').strip()


resumen = df.pivot_table(index='label', columns='split', values='filepath', aggfunc='count', fill_value=0).reindex(clases)
resumen = resumen[['train', 'val', 'test']]
resumen['total'] = resumen.sum(axis=1)
resumen['proporcion'] = resumen['total'] / resumen['total'].sum()
display(resumen)
print('Razón entre la clase mayoritaria y la minoritaria:', round(resumen['total'].max() / resumen['total'].min(), 2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
resumen[['train', 'val', 'test']].plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_title('Distribución de clases por conjunto')
ax.set_xlabel('')
ax.set_ylabel('Imágenes')
ax.set_xticklabels([nombre_corto(clase) for clase in resumen.index], rotation=45, ha='right')
ax.legend(title='Conjunto')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'balance_clases.png', dpi=150)
plt.show()

### 2.2 Muestras y propiedades

Las imágenes son de laboratorio, con una hoja centrada y un fondo relativamente uniforme. Esa uniformidad ayuda al modelo, pero no representa una fotografía de campo. La limitación se retoma en la auditoría y en las conclusiones.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, clase in zip(axes.ravel(), clases):
    fila = df[(df['label'] == clase) & (df['split'] == 'train')].iloc[0]
    ax.imshow(plt.imread(fila['abs_path']))
    ax.set_title(nombre_corto(clase), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'muestras_clases.png', dpi=150)
plt.show()

In [ ]:
propiedades = []
for ruta in df.sample(20, random_state=SEED)['abs_path']:
    imagen = plt.imread(ruta)
    propiedades.append({'forma': imagen.shape, 'minimo': float(imagen.min()), 'maximo': float(imagen.max())})
propiedades = pd.DataFrame(propiedades)
print('Formas encontradas:', propiedades['forma'].value_counts().to_dict())
display(propiedades.head())

Todas las imágenes revisadas son de 256x256 con tres canales. El modelo trabaja a 128x128 para reducir el costo de cómputo y mantener el detalle de las lesiones. La normalización se hace dentro del modelo con `Rescaling`, no en el dataset, para que el aumento de datos reciba valores comparables a los de la imagen original.

## 3. Preprocesamiento y aumento de datos

El preprocesamiento es mínimo para no introducir información externa. Redimensionamos a 128x128, mantenemos tres canales y normalizamos con la capa `Rescaling`. El aumento se aplica solo durante el entrenamiento.

Elegimos transformaciones que no cambian el significado de la hoja: volteo horizontal, rotación de 0,08, zoom de 0,10, traslación de 0,08 y contraste de 0,10. No usamos volteo vertical porque una hoja invertida no es una variación natural. No usamos recortes agresivos porque podrían eliminar lesiones pequeñas.

In [ ]:
def crear_aumento():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomTranslation(0.08, 0.08),
        tf.keras.layers.RandomContrast(0.10),
    ], name='aumento_datos')


data_augmentation = crear_aumento()
fila = df[df['split'] == 'train'].iloc[0]
imagen, _ = cargar_imagen(fila['abs_path'], fila['y'])
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes[0, 0].imshow(imagen.numpy().astype('uint8'))
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')
for ax in axes.ravel()[1:]:
    aumentada = data_augmentation(tf.expand_dims(imagen, 0), training=True)[0].numpy()
    ax.imshow(np.clip(aumentada, 0, 255).astype('uint8'))
    ax.set_title('Aumentada')
    ax.axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'aumento_datos.png', dpi=150)
plt.show()

El aumento genera copias plausibles y no reemplaza la falta de imágenes de campo. Su efecto se mide en la ablación. La capa se crea de nuevo para cada modelo para evitar que el estado de una corrida afecte a la siguiente.

## 4. Funciones del pipeline

El pipeline se organiza en funciones pequeñas y reutilizables. Cada etapa tiene una responsabilidad clara y se puede probar por separado. La tabla resume la relación entre las etapas del enunciado y las funciones del notebook.

| Etapa | Función | Propósito |
|---|---|---|
| 1. Carga | `cargar_datos` | leer split, verificar rutas y ordenar clases |
| 2. Análisis | celdas de EDA | balance, muestras y propiedades |
| 3. Preprocesamiento | `cargar_imagen`, `crear_dataset` | decodificar, redimensionar y normalizar |
| 4. Construcción | `construir_modelo` | MLP y CNN con regularización opcional |
| 5. Entrenamiento | `entrenar_modelo` | fit, parada temprana e historial |
| 6. Regularización | `crear_aumento`, `construir_modelo` | aumento, Dropout, BN y L2 |
| 7. Evaluación | `evaluar_metricas`, `graficar_curvas` | métricas, matriz y curvas |
| 8. Auditoría | `prueba_estres` | robustez a brillo y contraste |
| 9. Optimización | `optimizar_modelo`, `entrenar_final` | screening, búsqueda y semillas |
| XAI | `gradcam`, `explicar_lime` | mapas de activación y explicación local |

In [ ]:
def construir_modelo(tipo, filtros=(32, 64, 128), neuronas=128, capas_mlp=(512, 256, 128), dropout_conv=0.2, dropout_dense=0.3, l2=1e-4, usar_bn=True, usar_aumento=True, learning_rate=1e-4):
    regularizador = tf.keras.regularizers.l2(l2) if l2 and l2 > 0 else None
    capas = [tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)), tf.keras.layers.Rescaling(1./255)]
    if usar_aumento:
        capas.append(crear_aumento())
    if tipo == 'mlp':
        capas.append(tf.keras.layers.Flatten())
        for unidades in capas_mlp:
            capas.append(tf.keras.layers.Dense(unidades, kernel_initializer='he_normal', kernel_regularizer=regularizador))
            if usar_bn:
                capas.append(tf.keras.layers.BatchNormalization())
            capas.append(tf.keras.layers.Activation('relu'))
            capas.append(tf.keras.layers.Dropout(dropout_dense))
        capas.append(tf.keras.layers.Dense(len(clases), activation='softmax'))
    elif tipo == 'cnn':
        for i, filtros_capa in enumerate(filtros, start=1):
            capas.append(tf.keras.layers.Conv2D(filtros_capa, (3, 3), padding='same', kernel_initializer='he_normal', kernel_regularizer=regularizador, name=f'conv_{i}'))
            if usar_bn:
                capas.append(tf.keras.layers.BatchNormalization())
            capas.append(tf.keras.layers.Activation('relu', name=f'conv_{i}_relu'))
            capas.append(tf.keras.layers.MaxPooling2D((2, 2)))
            capas.append(tf.keras.layers.Dropout(dropout_conv))
        capas.append(tf.keras.layers.Flatten())
        capas.append(tf.keras.layers.Dense(neuronas, kernel_initializer='he_normal', kernel_regularizer=regularizador))
        if usar_bn:
            capas.append(tf.keras.layers.BatchNormalization())
        capas.append(tf.keras.layers.Activation('relu'))
        capas.append(tf.keras.layers.Dropout(dropout_dense))
        capas.append(tf.keras.layers.Dense(len(clases), activation='softmax'))
    else:
        raise ValueError(f'Tipo de modelo no valido: {tipo}')
    modelo = tf.keras.Sequential(capas, name=f'{tipo}_ep3')
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modelo


modelo_base = construir_modelo('cnn')
modelo_base.summary()
del modelo_base
tf.keras.backend.clear_session()

La CNN usa tres bloques de convolución con 32, 64 y 128 filtros, kernel 3x3, padding `same`, inicialización He y Batch Normalization opcional. Cada bloque termina en MaxPooling y Dropout. La cabeza tiene una capa densa de 128 unidades y una salida softmax de diez clases.

La MLP aplana la imagen y usa tres capas densas de 512, 256 y 128 unidades. Es una arquitectura mucho más grande en parámetros, pero no aprovecha la estructura espacial. Esa diferencia es la hipótesis central de la comparación.

In [ ]:
def entrenar_modelo(modelo, train_ds, val_ds, nombre, epochs=EPOCHS, patience=PACIENCIA, verbose=0):
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)]
    inicio = time.time()
    historia = modelo.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=verbose, shuffle=False)
    tiempo = time.time() - inicio
    pd.DataFrame(historia.history).to_csv(HISTORY_DIR / f'{nombre}.csv', index=False)
    return historia, tiempo


def graficar_curvas(historia, nombre):
    datos = historia.history if hasattr(historia, 'history') else historia
    epocas = range(1, len(datos['loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epocas, datos['loss'], label='Train', color='#1f77b4', linewidth=2)
    ax1.plot(epocas, datos['val_loss'], label='Validación', color='#ff7f0e', linewidth=2)
    ax1.set_title(f'Pérdida - {nombre}')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Pérdida')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax2.plot(epocas, datos['accuracy'], label='Train', color='#1f77b4', linewidth=2)
    ax2.plot(epocas, datos['val_accuracy'], label='Validación', color='#ff7f0e', linewidth=2)
    ax2.set_title(f'Accuracy - {nombre}')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / f'curvas_{nombre.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def evaluar_metricas(modelo, ds, nombre, guardar=True, graficar=True):
    y_true = np.concatenate([y_batch.numpy() for _, y_batch in ds], axis=0)
    y_prob = modelo.predict(ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    metricas = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    try:
        metricas['roc_auc'] = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    except ValueError:
        metricas['roc_auc'] = np.nan
    reporte = classification_report(y_true, y_pred, labels=np.arange(len(clases)), target_names=clases, output_dict=True, zero_division=0)
    reporte_df = pd.DataFrame(reporte).transpose()
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(clases)))
    if graficar:
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=clases, yticklabels=clases)
        plt.xlabel('Predicción')
        plt.ylabel('Clase real')
        plt.title(f'Matriz de confusión - {nombre}')
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.savefig(IMAGES_DIR / f'matriz_{nombre.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
        plt.show()
    if guardar:
        slug = nombre.lower().replace(' ', '_')
        (REPORTS_DIR / f'reporte_{slug}.txt').write_text(classification_report(y_true, y_pred, labels=np.arange(len(clases)), target_names=clases, zero_division=0))
        reporte_df.to_csv(REPORTS_DIR / f'reporte_{slug}.csv')
    return {
        'metricas': metricas,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'cm': cm,
        'reporte_df': reporte_df,
    }

In [ ]:
def optimizar_modelo(df, configuraciones, nombre_bloque, epochs=EPOCHS, verbose=0):
    resultados = []
    for i, config in enumerate(configuraciones, start=1):
        tf.keras.utils.set_random_seed(config.get('seed', SEED))
        train_ds = crear_dataset(df, 'train', entrenamiento=True, batch_size=config.get('batch_size', BATCH_SIZE))
        val_ds = crear_dataset(df, 'val', batch_size=config.get('batch_size', BATCH_SIZE))
        parametros = {k: v for k, v in config.items() if k not in {'nombre', 'batch_size', 'epochs', 'seed'}}
        modelo = construir_modelo(**parametros)
        historia, tiempo = entrenar_modelo(modelo, train_ds, val_ds, config['nombre'], epochs=config.get('epochs', epochs), verbose=verbose)
        evaluacion = evaluar_metricas(modelo, val_ds, config['nombre'], guardar=False, graficar=False)
        fila = {
            'bloque': nombre_bloque,
            'experimento': config['nombre'],
            'tipo': config['tipo'],
            'filtros': '-'.join(map(str, config.get('filtros', ()))),
            'neuronas': config.get('neuronas', 128),
            'dropout_conv': config.get('dropout_conv', 0.0),
            'dropout_dense': config.get('dropout_dense', 0.0),
            'l2': config.get('l2', 0.0),
            'usar_bn': config.get('usar_bn', False),
            'usar_aumento': config.get('usar_aumento', False),
            'learning_rate': config.get('learning_rate', 1e-4),
            'batch_size': config.get('batch_size', BATCH_SIZE),
            'accuracy_train': historia.history['accuracy'][-1],
            'accuracy_val': evaluacion['metricas']['accuracy'],
            'f1_macro_val': evaluacion['metricas']['f1_macro'],
            'brecha_val': historia.history['accuracy'][-1] - evaluacion['metricas']['accuracy'],
            'parametros': modelo.count_params(),
            'epocas': len(historia.history['loss']),
            'tiempo_s': tiempo,
        }
        resultados.append(fila)
        print(f"[{i}/{len(configuraciones)}] {config['nombre']}: F1 macro val {fila['f1_macro_val']:.4f}, accuracy val {fila['accuracy_val']:.4f}, {fila['epocas']} épocas, {tiempo:.1f} s")
        del modelo
        tf.keras.backend.clear_session()
    resultados_df = pd.DataFrame(resultados)
    resultados_df.to_csv(MODELS_DIR / f'experimentos_{nombre_bloque}.csv', index=False)
    mejor = resultados_df.sort_values('f1_macro_val', ascending=False).iloc[0]
    mejor_config = next(c for c in configuraciones if c['nombre'] == mejor['experimento'])
    return resultados_df, mejor_config


def entrenar_final(datos, config, nombre, semillas, archivo_modelo):
    resultados = []
    ultima_evaluacion = None
    ultimo_modelo = None
    for semilla in semillas:
        tf.keras.utils.set_random_seed(semilla)
        train_ds = crear_dataset(datos, 'train', entrenamiento=True, batch_size=config.get('batch_size', BATCH_SIZE))
        val_ds = crear_dataset(datos, 'val', batch_size=config.get('batch_size', BATCH_SIZE))
        test_ds = crear_dataset(datos, 'test', batch_size=config.get('batch_size', BATCH_SIZE))
        parametros = {k: v for k, v in config.items() if k not in {'nombre', 'batch_size', 'epochs', 'seed'}}
        modelo = construir_modelo(**parametros)
        nombre_corrida = f'{nombre}_semilla_{semilla}'
        historia, tiempo = entrenar_modelo(modelo, train_ds, val_ds, nombre_corrida, epochs=config.get('epochs', EPOCHS), verbose=2)
        evaluacion = evaluar_metricas(modelo, test_ds, nombre_corrida, guardar=(semilla == semillas[0]), graficar=(semilla == semillas[0]))
        evaluacion_val = evaluar_metricas(modelo, val_ds, f'{nombre_corrida}_val', guardar=False, graficar=False)
        evaluacion_train = evaluar_metricas(modelo, train_ds, f'{nombre_corrida}_train', guardar=False, graficar=False)
        if semilla == semillas[0]:
            graficar_curvas(historia, nombre_corrida)
            modelo.optimizer = None
            modelo.save(MODELS_DIR / archivo_modelo)
            ultima_evaluacion = evaluacion
            ultimo_modelo = modelo
        resultados.append({
            'modelo': nombre,
            'semilla': semilla,
            'tipo': config.get('tipo'),
            'filtros': '-'.join(map(str, config.get('filtros', ()))),
            'neuronas': config.get('neuronas', 128),
            'dropout_conv': config.get('dropout_conv', 0.0),
            'dropout_dense': config.get('dropout_dense', 0.0),
            'l2': config.get('l2', 0.0),
            'usar_bn': config.get('usar_bn', False),
            'usar_aumento': config.get('usar_aumento', False),
            'learning_rate': config.get('learning_rate', 1e-4),
            'batch_size': config.get('batch_size', BATCH_SIZE),
            'accuracy_train': evaluacion_train['metricas']['accuracy'],
            'accuracy_val': evaluacion_val['metricas']['accuracy'],
            'f1_macro_val': evaluacion_val['metricas']['f1_macro'],
            'accuracy': evaluacion['metricas']['accuracy'],
            'balanced_accuracy': evaluacion['metricas']['balanced_accuracy'],
            'f1_macro': evaluacion['metricas']['f1_macro'],
            'f1_weighted': evaluacion['metricas']['f1_weighted'],
            'roc_auc': evaluacion['metricas']['roc_auc'],
            'brecha_test': evaluacion_train['metricas']['accuracy'] - evaluacion['metricas']['accuracy'],
            'parametros': modelo.count_params(),
            'epocas': len(historia.history['loss']),
            'tiempo_s': tiempo,
        })
        print(f'{nombre} semilla {semilla}: accuracy {evaluacion["metricas"]["accuracy"]:.4f}, F1 macro {evaluacion["metricas"]["f1_macro"]:.4f}, {len(historia.history["loss"])} épocas, {tiempo:.1f} s')
        if semilla != semillas[0]:
            del modelo
        tf.keras.backend.clear_session()
    return pd.DataFrame(resultados), ultima_evaluacion, ultimo_modelo

In [ ]:
def gradcam(modelo, imagen, clase_idx, capa='conv_3_relu'):
    grad_model = tf.keras.Model(inputs=modelo.inputs[0], outputs=[modelo.get_layer(capa).output, modelo.outputs[0]])
    with tf.GradientTape() as tape:
        conv_outputs, predicciones = grad_model(imagen)
        class_channel = predicciones[:, clase_idx]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def superponer_gradcam(imagen, heatmap):
    heatmap = tf.image.resize(heatmap[..., None], IMG_SIZE).numpy().squeeze()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    jet = plt.cm.jet(heatmap)[..., :3]
    return np.clip(imagen / 255.0 * 0.55 + jet * 0.45, 0, 1)


def explicar_lime(modelo, imagen, num_samples=300):
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    explainer = lime_image.LimeImageExplainer()
    explicacion = explainer.explain_instance(
        np.asarray(imagen[0]).astype('uint8'),
        modelo.predict,
        top_labels=1,
        hide_color=0,
        num_samples=num_samples,
    )
    temp, mask = explicacion.get_image_and_mask(
        explicacion.top_labels[0],
        positive_only=True,
        num_features=5,
        hide_rest=False,
    )
    return mark_boundaries(temp / 255.0, mask), explicacion


def medir_inferencia(modelo, ds, repeticiones=3):
    for x_batch, _ in ds.take(1):
        modelo.predict(x_batch, verbose=0)
        tiempos = []
        for _ in range(repeticiones):
            inicio = time.time()
            modelo.predict(x_batch, verbose=0)
            tiempos.append((time.time() - inicio) / len(x_batch) * 1000)
    return float(np.mean(tiempos))


def ajustar_brillo(x, delta):
    return tf.clip_by_value(x + delta, 0.0, 255.0)


def ajustar_contraste(x, factor):
    return tf.clip_by_value((x / 255.0 - 0.5) * factor + 0.5, 0.0, 1.0) * 255.0


def prueba_estres(modelo, df, nombre):
    transformaciones = {
        'brillo_bajo': lambda x: ajustar_brillo(x, -30.0),
        'brillo_alto': lambda x: ajustar_brillo(x, 30.0),
        'contraste_bajo': lambda x: ajustar_contraste(x, 0.7),
        'contraste_alto': lambda x: ajustar_contraste(x, 1.3),
    }
    filas = []
    for nombre_transformacion, transformacion in transformaciones.items():
        ds = crear_dataset(df, 'test', transformacion=transformacion)
        evaluacion = evaluar_metricas(modelo, ds, f'estres_{nombre}_{nombre_transformacion}', guardar=False, graficar=False)
        filas.append({
            'transformacion': nombre_transformacion,
            'accuracy': evaluacion['metricas']['accuracy'],
            'f1_macro': evaluacion['metricas']['f1_macro'],
        })
    return pd.DataFrame(filas)

Las funciones de entrenamiento y evaluación comparten la misma partición, la misma semilla y el mismo preprocesamiento. La búsqueda evalúa siempre en validación. El test se reserva para la evaluación final y para la auditoría. La parada temprana monitorea `val_loss` y restaura los mejores pesos.

## 5. Referencia de la EP2

La EP2 comparó una MLP de 512, 256 y 128 unidades con una CNN de 32, 64 y 128 filtros, sin aumento de datos. La CNN obtuvo 94,1% de accuracy y 93,2% de F1 macro. La MLP obtuvo 71,5% y 64,6%. Esa brecha es el punto de partida.

En la EP3 no repetimos la EP2. Usamos sus resultados como referencia y entrenamos variantes optimizadas con el mismo split. La comparación final es entre la mejor MLP y la mejor CNN de este notebook.

In [ ]:
referencia_ep2 = pd.DataFrame([
    {'modelo': 'MLP EP2', 'accuracy': 0.715, 'f1_macro': 0.646, 'parametros': 25331850, 'tiempo_s': 120.1},
    {'modelo': 'CNN EP2', 'accuracy': 0.941, 'f1_macro': 0.932, 'parametros': 3305930, 'tiempo_s': 390.6},
])
display(referencia_ep2)

## 6. Ablación de regularización

La ablación mide el efecto de cada técnica por separado. Partimos de una CNN sin Dropout, sin Batch Normalization, sin L2 y sin aumento. Luego agregamos una técnica a la vez. La métrica de decisión es el F1 macro de validación.

Esta etapa responde a la pregunta de qué técnica aporta y cuál no. No buscamos el mejor resultado absoluto, buscamos entender el efecto de cada componente antes de combinarlos.

In [ ]:
config_base = {
    'tipo': 'cnn',
    'filtros': (32, 64, 128),
    'neuronas': 128,
    'dropout_conv': 0.2,
    'dropout_dense': 0.3,
    'l2': 1e-4,
    'usar_bn': True,
    'usar_aumento': True,
    'learning_rate': 1e-4,
    'batch_size': 32,
    'seed': SEED,
}

ablacion = [
    {**config_base, 'nombre': 'A1_sin_regularizacion', 'dropout_conv': 0.0, 'dropout_dense': 0.0, 'l2': 0.0, 'usar_bn': False, 'usar_aumento': False},
    {**config_base, 'nombre': 'A2_dropout', 'dropout_conv': 0.2, 'dropout_dense': 0.3, 'l2': 0.0, 'usar_bn': False, 'usar_aumento': False},
    {**config_base, 'nombre': 'A3_batch_norm', 'dropout_conv': 0.0, 'dropout_dense': 0.0, 'l2': 0.0, 'usar_bn': True, 'usar_aumento': False},
    {**config_base, 'nombre': 'A4_l2', 'dropout_conv': 0.0, 'dropout_dense': 0.0, 'l2': 1e-4, 'usar_bn': False, 'usar_aumento': False},
    {**config_base, 'nombre': 'A5_aumento', 'dropout_conv': 0.0, 'dropout_dense': 0.0, 'l2': 0.0, 'usar_bn': False, 'usar_aumento': True},
    {**config_base, 'nombre': 'A6_mlp_base', 'tipo': 'mlp', 'filtros': (), 'capas_mlp': (512, 256, 128), 'dropout_conv': 0.0, 'dropout_dense': 0.2, 'l2': 0.0, 'usar_bn': False, 'usar_aumento': False},
]

resultados_ablacion, _ = optimizar_modelo(df, ablacion, 'ablacion', epochs=EPOCHS)
display(resultados_ablacion.sort_values('f1_macro_val', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=resultados_ablacion, x='experimento', y='f1_macro_val', color='#1f77b4', ax=ax)
ax.set_ylim(0, 1)
ax.set_title('F1 macro de validación por configuración')
ax.set_xlabel('')
ax.set_ylabel('F1 macro')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'ablacion_f1.png', dpi=150)
plt.show()

### 6.1 Interpretación

{{ABLACION_TEXTO}}

## 7. Screening y búsqueda de hiperparámetros

El screening mueve un hiperparámetro a la vez alrededor de una configuración regularizada. Revisamos tasa de aprendizaje, tamaño de lote, Dropout convolucional, Dropout denso, cantidad de filtros y L2. La configuración con mejor F1 macro de validación pasa a la búsqueda.

La búsqueda principal es Random Search. Elegimos este método porque cubre un espacio amplio con un presupuesto fijo, es reproducible y no requiere una librería adicional. Una grilla completa con seis hiperparámetros y tres valores por dimensión tendría 729 combinaciones. La validación cruzada de `GridSearchCV` multiplicaría el costo por cada fold y está pensada para estimadores de scikit-learn, no para un modelo Keras de este tamaño.

In [ ]:
config_regularizada = {
    'tipo': 'cnn',
    'filtros': (32, 64, 128),
    'neuronas': 128,
    'dropout_conv': 0.2,
    'dropout_dense': 0.3,
    'l2': 1e-4,
    'usar_bn': True,
    'usar_aumento': True,
    'learning_rate': 1e-4,
    'batch_size': 32,
    'seed': SEED,
}

screening = [
    {**config_regularizada, 'nombre': 'B1_base_regularizada'},
    {**config_regularizada, 'nombre': 'B2_lr_1e-3', 'learning_rate': 1e-3},
    {**config_regularizada, 'nombre': 'B3_batch_64', 'batch_size': 64},
    {**config_regularizada, 'nombre': 'B4_dropout_conv_0.3', 'dropout_conv': 0.3},
    {**config_regularizada, 'nombre': 'B5_dropout_dense_0.4', 'dropout_dense': 0.4},
    {**config_regularizada, 'nombre': 'B6_filtros_64_128_256', 'filtros': (64, 128, 256)},
    {**config_regularizada, 'nombre': 'B7_l2_1e-5', 'l2': 1e-5},
]

resultados_screening, mejor_screening = optimizar_modelo(df, screening, 'screening', epochs=EPOCHS)
display(resultados_screening.sort_values('f1_macro_val', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=resultados_screening, x='experimento', y='f1_macro_val', color='#1f77b4', ax=ax)
ax.set_ylim(0, 1)
ax.set_title('Screening de hiperparámetros')
ax.set_xlabel('')
ax.set_ylabel('F1 macro de validación')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'screening_f1.png', dpi=150)
plt.show()

### 7.1 Interpretación del screening

{{SCREENING_TEXTO}}

In [ ]:
rng = np.random.default_rng(SEED)
configuraciones_busqueda = [{**mejor_screening, 'nombre': 'C1_mejor_screening', 'seed': SEED}]
vistos = set()
while len(configuraciones_busqueda) < 8:
    config = {
        'tipo': 'cnn',
        'filtros': [(32, 64, 128), (64, 128, 256)][int(rng.integers(0, 2))],
        'neuronas': 128,
        'dropout_conv': float(rng.uniform(0.2, 0.4)),
        'dropout_dense': float(rng.uniform(0.3, 0.5)),
        'l2': float(10 ** rng.uniform(-5, -3)),
        'usar_bn': True,
        'usar_aumento': True,
        'learning_rate': float(10 ** rng.uniform(-4, -3)),
        'batch_size': int(rng.choice([32, 64])),
        'seed': SEED,
    }
    clave = (config['filtros'], round(config['dropout_conv'], 3), round(config['dropout_dense'], 3), round(config['l2'], 6), round(config['learning_rate'], 6), config['batch_size'])
    if clave in vistos:
        continue
    vistos.add(clave)
    config['nombre'] = f'C{len(configuraciones_busqueda) + 1}_aleatorio'
    configuraciones_busqueda.append(config)

tabla_busqueda = pd.DataFrame([
    {
        'nombre': c['nombre'],
        'filtros': '-'.join(map(str, c['filtros'])),
        'dropout_conv': round(c['dropout_conv'], 3),
        'dropout_dense': round(c['dropout_dense'], 3),
        'l2': round(c['l2'], 6),
        'learning_rate': round(c['learning_rate'], 6),
        'batch_size': c['batch_size'],
    }
    for c in configuraciones_busqueda
])
display(tabla_busqueda)

In [ ]:
resultados_busqueda, mejor_busqueda = optimizar_modelo(df, configuraciones_busqueda, 'busqueda', epochs=EPOCHS)
display(resultados_busqueda.sort_values('f1_macro_val', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=resultados_busqueda, x='experimento', y='f1_macro_val', color='#1f77b4', ax=ax)
ax.set_ylim(0, 1)
ax.set_title('Random Search')
ax.set_xlabel('')
ax.set_ylabel('F1 macro de validación')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'busqueda_f1.png', dpi=150)
plt.show()

### 7.2 Interpretación de la búsqueda

{{BUSQUEDA_TEXTO}}

## 8. Entrenamiento final y evaluación

La configuración ganadora se entrena con tres semillas para reportar media y desviación. La semilla 42 se guarda como modelo final. El conjunto de test se usa solo para reportar el desempeño final, no para elegir la configuración.

La MLP se optimiza por separado. Comparamos la MLP de la EP2, una versión regularizada y una versión más liviana. La mejor por F1 macro de validación se entrena con tres semillas y se guarda como modelo final.

In [ ]:
resultados_cnn, evaluacion_cnn, modelo_cnn = entrenar_final(df, mejor_busqueda, 'CNN_final', [42, 43, 44], 'cnn_ep3_final.keras')
display(resultados_cnn)

In [ ]:
reporte_cnn = evaluacion_cnn['reporte_df'].loc[clases, ['precision', 'recall', 'f1-score', 'support']]
reporte_cnn.insert(0, 'clase', [nombre_corto(clase) for clase in reporte_cnn.index])
display(reporte_cnn.sort_values('recall'))

### 8.1 Interpretación de la CNN final

{{CNN_TEXTO}}

In [ ]:
config_mlp = {
    'tipo': 'mlp',
    'filtros': (),
    'capas_mlp': (512, 256, 128),
    'dropout_dense': 0.2,
    'l2': 0.0,
    'usar_bn': False,
    'usar_aumento': False,
    'learning_rate': 1e-4,
    'batch_size': 32,
    'seed': SEED,
}

configuraciones_mlp = [
    {**config_mlp, 'nombre': 'E1_mlp_ep2'},
    {**config_mlp, 'nombre': 'E2_mlp_regularizada', 'dropout_dense': 0.3, 'l2': 1e-4, 'usar_bn': True, 'usar_aumento': True},
    {**config_mlp, 'nombre': 'E3_mlp_ligera', 'capas_mlp': (256, 128, 64), 'dropout_dense': 0.3, 'l2': 1e-4, 'usar_bn': True, 'usar_aumento': True},
]

resultados_mlp_busqueda, mejor_mlp = optimizar_modelo(df, configuraciones_mlp, 'mlp', epochs=30)
display(resultados_mlp_busqueda.sort_values('f1_macro_val', ascending=False))

In [ ]:
resultados_mlp, evaluacion_mlp, modelo_mlp = entrenar_final(df, mejor_mlp, 'MLP_final', [42, 43, 44], 'mlp_ep3_final.keras')
display(resultados_mlp)

In [ ]:
reporte_mlp = evaluacion_mlp['reporte_df'].loc[clases, ['precision', 'recall', 'f1-score', 'support']]
reporte_mlp.insert(0, 'clase', [nombre_corto(clase) for clase in reporte_mlp.index])
display(reporte_mlp.sort_values('recall'))

### 8.2 Interpretación de la MLP final

{{MLP_TEXTO}}

In [ ]:
filas = []
for nombre, resultados, evaluacion, modelo in [
    ('CNN final', resultados_cnn, evaluacion_cnn, modelo_cnn),
    ('MLP final', resultados_mlp, evaluacion_mlp, modelo_mlp),
]:
    filas.append({
        'modelo': nombre,
        'accuracy': evaluacion['metricas']['accuracy'],
        'balanced_accuracy': evaluacion['metricas']['balanced_accuracy'],
        'precision_macro': evaluacion['metricas']['precision_macro'],
        'recall_macro': evaluacion['metricas']['recall_macro'],
        'f1_macro': evaluacion['metricas']['f1_macro'],
        'f1_weighted': evaluacion['metricas']['f1_weighted'],
        'roc_auc': evaluacion['metricas']['roc_auc'],
        'parametros': modelo.count_params(),
        'brecha_test': resultados.loc[resultados['semilla'] == 42, 'brecha_test'].iloc[0],
        'tiempo_entrenamiento_s': resultados.loc[resultados['semilla'] == 42, 'tiempo_s'].iloc[0],
        'tiempo_inferencia_ms': medir_inferencia(modelo, test_ds),
        'f1_macro_media': resultados['f1_macro'].mean(),
        'f1_macro_desviacion': resultados['f1_macro'].std(),
    })
comparacion = pd.DataFrame(filas)
display(comparacion)
comparacion.to_csv(MODELS_DIR / 'comparacion_final.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
comparacion.set_index('modelo')[['accuracy', 'f1_macro']].plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e'])
ax.set_ylim(0, 1)
ax.set_title('Comparación de métricas en test')
ax.set_xlabel('')
ax.set_ylabel('Valor')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'comparacion_modelos.png', dpi=150)
plt.show()

### 8.3 Interpretación de la comparación

{{COMPARACION_TEXTO}}

## 9. Auditoría de sesgos

La auditoría revisa tres frentes. El sesgo de entrada mira el desbalance y la representación. El sesgo de sistema mira el desempeño por clase, la brecha entre métricas macro y ponderadas y los errores. El sesgo de aplicación mira el uso fuera del laboratorio y el costo de un falso negativo.

No buscamos demostrar que el modelo es justo. Buscamos identificar dónde puede fallar y qué medidas reducirían el riesgo.

In [ ]:
auditoria_clases = reporte_cnn.copy()
auditoria_clases['brecha_contra_mejor'] = auditoria_clases['recall'].max() - auditoria_clases['recall']
display(auditoria_clases.sort_values('recall'))
print('F1 macro:', round(evaluacion_cnn['metricas']['f1_macro'], 4))
print('F1 ponderado:', round(evaluacion_cnn['metricas']['f1_weighted'], 4))
print('Brecha macro y ponderado:', round(evaluacion_cnn['metricas']['f1_weighted'] - evaluacion_cnn['metricas']['f1_macro'], 4))

In [ ]:
matriz = evaluacion_cnn['cm']
indice_healthy = clases.index('Tomato_healthy')
falsos_negativos = int(matriz[indice_healthy].sum() - matriz[indice_healthy, indice_healthy])
print('Casos sanos mal clasificados como enfermedad:', falsos_negativos)
print('Total de casos sanos en test:', int(matriz[indice_healthy].sum()))

In [ ]:
confianza = evaluacion_cnn['y_prob'].max(axis=1)
aciertos = evaluacion_cnn['y_pred'] == evaluacion_cnn['y_true']
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(confianza[aciertos], bins=20, alpha=0.6, label='Aciertos', color='#1f77b4')
ax.hist(confianza[~aciertos], bins=20, alpha=0.6, label='Errores', color='#ff7f0e')
ax.set_xlabel('Confianza de la predicción')
ax.set_ylabel('Casos')
ax.set_title('Distribución de confianza en test')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'confianza_cnn.png', dpi=150)
plt.show()
print('Confianza media en aciertos:', round(float(confianza[aciertos].mean()), 4))
print('Confianza media en errores:', round(float(confianza[~aciertos].mean()), 4))

In [ ]:
import hashlib
hashes = df['abs_path'].map(lambda ruta: hashlib.md5(Path(ruta).read_bytes()).hexdigest())
print('Imágenes duplicadas exactas:', int(hashes.duplicated().sum()))
print('Clases con menos de 400 imágenes en total:', resumen[resumen['total'] < 400].index.tolist())

In [ ]:
estres = prueba_estres(modelo_cnn, df, 'cnn')
display(estres)
estres.to_csv(MODELS_DIR / 'prueba_estres.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=estres, x='transformacion', y='f1_macro', color='#1f77b4', ax=ax)
ax.set_ylim(0, 1)
ax.set_title('F1 macro bajo cambios de brillo y contraste')
ax.set_xlabel('')
ax.set_ylabel('F1 macro')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'prueba_estres.png', dpi=150)
plt.show()

### 9.1 Interpretación de la auditoría

{{AUDITORIA_TEXTO}}

### 9.2 Medidas de mitigación

| Fuente | Medida | Efecto esperado |
|---|---|---|
| Desbalance | recolectar más imágenes de mosaic virus y Early blight | mejorar el recall de clases minoritarias |
| Laboratorio | validar con fotografías de campo | medir la caída real de generalización |
| Falsos negativos | revisión humana de casos de baja confianza | reducir el riesgo agronómico |
| Cambios de población | monitoreo y reentrenamiento periódico | mantener la calibración |
| Uso fuera de alcance | documentar límites y supervisión | evitar decisiones automáticas |

## 10. Explicabilidad

Usamos Grad-CAM en la CNN y LIME en la MLP. Grad-CAM muestra qué regiones de la imagen activan la clase predicha. LIME muestra qué superpíxeles empujan la predicción. Las dos técnicas son locales y no prueban causalidad, pero ayudan a detectar atajos y errores.

Revisamos aciertos y errores. Un mapa que mira el fondo o el borde de la hoja es una señal de alerta. Un mapa que se concentra en la lesión es una señal de apoyo, no una garantía.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, clase in zip(axes.ravel(), clases):
    fila = df[(df['label'] == clase) & (df['split'] == 'test')].iloc[0]
    imagen, _ = cargar_imagen(fila['abs_path'], fila['y'])
    imagen = tf.expand_dims(imagen, 0)
    prediccion = int(tf.argmax(modelo_cnn.predict(imagen, verbose=0), axis=1)[0])
    heatmap = gradcam(modelo_cnn, imagen, prediccion)
    ax.imshow(superponer_gradcam(imagen[0].numpy(), heatmap))
    ax.set_title(f'{nombre_corto(clase)}: {prediccion == fila["y"]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'gradcam_clases.png', dpi=150)
plt.show()

In [ ]:
test_df = df[df['split'] == 'test'].reset_index(drop=True)
errores = np.where(evaluacion_cnn['y_pred'] != evaluacion_cnn['y_true'])[0]
seleccion = errores[:5]
fig, axes = plt.subplots(1, max(len(seleccion), 1), figsize=(15, 3))
for ax, indice in zip(np.atleast_1d(axes), seleccion):
    fila = test_df.iloc[indice]
    imagen, _ = cargar_imagen(fila['abs_path'], fila['y'])
    imagen = tf.expand_dims(imagen, 0)
    prediccion = int(evaluacion_cnn['y_pred'][indice])
    heatmap = gradcam(modelo_cnn, imagen, prediccion)
    ax.imshow(superponer_gradcam(imagen[0].numpy(), heatmap))
    ax.set_title(f'Real: {nombre_corto(fila["label"])}\nPred: {nombre_corto(clases[prediccion])}', fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'gradcam_errores.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, clase in zip(axes, ['Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_healthy']):
    fila = df[(df['label'] == clase) & (df['split'] == 'test')].iloc[0]
    imagen, _ = cargar_imagen(fila['abs_path'], fila['y'])
    imagen = tf.expand_dims(imagen, 0)
    temp, _ = explicar_lime(modelo_mlp, imagen, num_samples=200)
    ax.imshow(temp)
    ax.set_title(nombre_corto(clase), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'lime_mlp.png', dpi=150)
plt.show()

### 10.1 Interpretación de la explicabilidad

{{XAI_TEXTO}}

## 11. Conclusiones, limitaciones y trabajo futuro

{{CONCLUSIONES_TEXTO}}

### 11.1 Limitaciones

- Las imágenes son de laboratorio y no representan todas las condiciones de campo.
- El desbalance favorece a las clases mayoritarias.
- La selección se hizo con una sola partición de validación.
- Grad-CAM y LIME son explicaciones locales y aproximadas.
- El modelo no reemplaza el diagnóstico de un especialista.

### 11.2 Trabajo futuro

- Validar con imágenes de campo y de otras variedades de tomate.
- Probar transfer learning con modelos preentrenados.
- Explorar calibración de probabilidades y umbrales por clase.
- Incorporar monitoreo de deriva en producción.
- Ampliar la muestra de clases minoritarias.

### 11.3 Referencias

- Bishop, C. y Bishop, H. (2023). Deep Learning: Foundations and Concepts.
- Goodfellow, I., Bengio, Y. y Courville, A. (2016). Deep Learning. MIT Press.
- Prince, S. (2023). Understanding Deep Learning. MIT Press.
- Chollet, F. (2021). Deep Learning with Python. Manning.
- Selvaraju, R. et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization.
- Ribeiro, M., Singh, S. y Guestrin, C. (2016). Why Should I Trust You? Explaining the Predictions of Any Classifier.
- PlantVillage. Dataset de imágenes de hojas.

In [ ]:
experimentos = pd.concat([
    resultados_ablacion,
    resultados_screening,
    resultados_busqueda,
    resultados_cnn.assign(bloque='final_cnn'),
    resultados_mlp_busqueda.assign(bloque='busqueda_mlp'),
    resultados_mlp.assign(bloque='final_mlp'),
], ignore_index=True)
experimentos.to_csv(MODELS_DIR / 'experimentos.csv', index=False)


def a_json(valor):
    if isinstance(valor, dict):
        return {k: a_json(v) for k, v in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [a_json(v) for v in valor]
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating, float)):
        if np.isnan(valor):
            return None
        return float(valor)
    if isinstance(valor, np.ndarray):
        return a_json(valor.tolist())
    return valor


metricas_finales = {
    'cnn': {
        'config': mejor_busqueda,
        'test': evaluacion_cnn['metricas'],
        'semillas': resultados_cnn[['semilla', 'accuracy', 'f1_macro']].to_dict(orient='records'),
        'parametros': int(modelo_cnn.count_params()),
    },
    'mlp': {
        'config': mejor_mlp,
        'test': evaluacion_mlp['metricas'],
        'semillas': resultados_mlp[['semilla', 'accuracy', 'f1_macro']].to_dict(orient='records'),
        'parametros': int(modelo_mlp.count_params()),
    },
    'comparacion': comparacion.to_dict(orient='records'),
}
with open(MODELS_DIR / 'metricas_finales.json', 'w', encoding='utf-8') as archivo:
    json.dump(a_json(metricas_finales), archivo, indent=2, ensure_ascii=False)

print('Archivos generados')
for ruta in [MODELS_DIR / 'experimentos.csv', MODELS_DIR / 'metricas_finales.json', MODELS_DIR / 'comparacion_final.csv', MODELS_DIR / 'cnn_ep3_final.keras', MODELS_DIR / 'mlp_ep3_final.keras']:
    print(ruta.relative_to(RAIZ), round(ruta.stat().st_size / 1e6, 2), 'MB')

## 12. Cierre

El notebook deja los artefactos principales en `models/` y las figuras en `images/`. El informe técnico amplía cada sección en `INFORME_TECNICO.md`. La defensa y la presentación se apoyan en `DEFENSA_EP3.md` y `docs/presentacion.md`.